module  {
  func.func @foo(%arg0: i32) -> i32 {
    %0 = arith.index_cast %arg0: i32 to index
    %c0_i32 = arith.constant 0: i32
    %c1_i32 = arith.constant 1: i32
    %c2_i32 = arith.constant 2: i32
    %1 = scf.index_switch %0 -> i32
    case 0 {
      scf.yield %c0_i32: i32
    }
    default {
      scf.yield %c1_i32: i32
    }
    %2 = arith.shli %1, %c2_i32: i32
    return %2: i32
  }
}

// should expect the following output
// func.func @foo(%arg0: i32) -> i32 {
//   %c0_i32 = arith.constant 0: i32
//   %c4_i32 = arith.constant 4: i32
//   %0 = arith.cmpi eq %arg0, %c0_i32: i32
//   %1 = arith.select %0, %c0_i32, %c4_i32: i32
//   return %1: i32
// }


In [ ]:
#include "mlir/IR/Dialect.h"
#include "mlir/InitAllDialects.h"
#include "mlir/InitAllPasses.h"
#include "mlir/Pass/PassRegistry.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Dialect/SCF/IR/SCF.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Tools/mlir-opt/MlirOptMain.h"
#include "mlir/Transforms/GreedyPatternRewriteDriver.h"
#include "mlir/IR/PatternMatch.h"

// docs: https://mlir.llvm.org/docs/PatternRewriter/
// arith dialect: https://mlir.llvm.org/docs/Dialects/ArithOps/
// scf dialect: https://mlir.llvm.org/docs/Dialects/SCFDialect/
// examples: https://github.com/llvm/llvm-project/tree/main/mlir/examples

namespace mlir {

class InstCombinePass : public PassWrapper<InstCombinePass, OperationPass<func::FuncOp>> {
    StringRef getArgument() const final {
        return "instcombine";
    }

    StringRef getDescription() const final {
        return "A simple pass to optimize some scf and arith operations";
    }

    void runOnOperation() {
        // TODO: please implement logic here 
        // *BEFORE*
        // 1. getting index; 
        // 2. assign 3 variables 0,1,2 
        // 3. use the index as index_switch(for the case...)
        // 4. if index=0 give 0, else 1 
        // 5. then we shift left the case output by 2 and return this value
        //ESSENTIALLY WE RETURN 0 WHEN arg0==0 else 4

        //*AFTER* -- WE HAVE TO RETURN 0 WHEN arg0==0 else 4
        //1. assign 2 constants 0,4(note shift left by 2 is actually multiplying a number by 4)
        //2. compare values of arg0 and 0. cmpi gives 1 if eq
        //3. if it is equal we return 0 else 4..select sees if %0 ==1 then return\
        second value(0) else return third value(equals 4)

        // *STEPS*
        // remove subsequent arith.constant whose argument value non-zero
        // add c4_i32 as new arith.constant equal to 4
        // replace scf.index_switch and all cases with arith.cmpi eq 
        // remove shli
        // insert arith.select that outputs either 0 or 4 based on output of arith.cmpi

        func::FuncOp func= getOperation();
        auto arg0= func.getArguments()[0]; //stores arg0

        //here we walk over every operation defined inside our func and check if its a switchop
        func.walk([&](scf::IndexSwitchOp op){
            //first we need to store c0_i32
            auto selector= op.getArg(); //get value of arg0 (here %0)
            auto loc=op.getLoc();
            //auto caseVal= op.getCaseValues()[0]; //int value of ith case i.e. 0 here
     
            auto& case0Reg= op.getCaseRegions()[0].front();//return body block of case0
            auto& defaultReg= op.getDefaultBlock();//return body block of default
            auto outTrue=(dyn_cast<scf::YieldOp>(case0Reg.getTerminator())).getOperand(0); //access the scf.yield in this block and we save c0_i32
            auto outFalse=(dyn_cast<scf::YieldOp>(defaultReg.getTerminator())).getOperand(0); //save c1_i32

            //now at this location we want to insert cmpi and select with all info we gathered
            OpBuilder builder(op);
            auto fourConst=builder.create<arith::ConstantIntOp>(loc, 4,32);
            auto cmpi= builder.create<arith::CmpIOp>(loc,arith::CmpIPredicate::eq, arg0,outTrue);
            auto select= builder.create<arith::SelectOp>(loc,cmpi.getResult(),outTrue,fourConst);

            op.getResult(0).replaceAllUsesWith(select.getResult());
            op.erase();

        });
        //so we replace shli with new arith.constant(4) instruction
        func.walk([&](arith::ShLIOp shliOp){
            Value LHS= shliOp.getLhs();
            Value RHS= shliOp.getRhs();
            auto firstArg= LHS.getDefiningOp<scf::IndexSwitchOp>(); //check is first arg is from output of indexswitch op
            auto shiftAmt= RHS.getDefiningOp<arith::ConstantIntOp>().value();
            auto locShli= shliOp.getLoc();
            OpBuilder builder(shliOp);

            //auto fourConst=builder.create<arith::ConstantIntOp>(locShli,4,32);
            shliOp.getResult().replaceAllUsesWith(LHS);
            shliOp.erase();
            

        });
        //now we have to remove all arith.indexcast not used
        func.walk([&](arith::IndexCastOp castOp){
            if (castOp.use_empty()){
                castOp.erase();
            }
        });
        //now we have to remove all arith.constant not used
        func.walk([&](arith::ConstantOp constOp){
            if (constOp.use_empty()){
                constOp.erase();
            }
        });
};

};
}

int main(int argc, char **argv) {
  mlir::registerAllPasses();
  mlir::PassRegistration<mlir::InstCombinePass>();

  mlir::DialectRegistry registry;
  registerAllDialects(registry);

  return mlir::asMainReturnCode(
      mlir::MlirOptMain(argc, argv, "Custom optimizer driver\n", registry));
}


In [ ]:

/// # Attempt 2
#include "mlir/IR/Dialect.h"
#include "mlir/InitAllDialects.h"
#include "mlir/InitAllPasses.h"
#include "mlir/Pass/PassRegistry.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Dialect/SCF/IR/SCF.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Tools/mlir-opt/MlirOptMain.h"
#include "mlir/Transforms/GreedyPatternRewriteDriver.h"
#include "mlir/IR/PatternMatch.h"

// docs: https://mlir.llvm.org/docs/PatternRewriter/
// arith dialect: https://mlir.llvm.org/docs/Dialects/ArithOps/
// scf dialect: https://mlir.llvm.org/docs/Dialects/SCFDialect/
// examples: https://github.com/llvm/llvm-project/tree/main/mlir/examples

namespace mlir {

class InstCombinePass : public PassWrapper<InstCombinePass, OperationPass<func::FuncOp>> {
    StringRef getArgument() const final {
        return "instcombine";
    }

    StringRef getDescription() const final {
        return "A simple pass to optimize some scf and arith operations";
    }

    void runOnOperation() {
        // TODO: please implement logic here 
        // *BEFORE*
        // 1. getting index; 
        // 2. assign 3 variables 0,1,2 
        // 3. use the index as index_switch(for the case...)
        // 4. if index=0 give 0, else 1 
        // 5. then we shift left the case output by 2 and return this value
        //ESSENTIALLY WE RETURN 0 WHEN arg0==0 else 4

        //*AFTER* -- WE HAVE TO RETURN 0 WHEN arg0==0 else 4
        //1. assign 2 constants 0,4(note shift left by 2 is actually multiplying a number by 4)
        //2. compare values of arg0 and 0. cmpi gives 1 if eq
        //3. if it is equal we return 0 else 4..select sees if %0 ==1 then return\
        second value(0) else return third value(equals 4)

        //**ATTEMPT2**
        // 1. getting index; 
        // 2. assign 3 variables 0,1,2 
        // 3. use the index as index_switch(for the case...)
        // 4. if index=0 give 0, else output input of user into func
        // 5. then we shift left the case output by 2 and return this value
        //ESSENTIALLY WE RETURN 0 WHEN arg0==0 else arg0*4

        // *STEPS*
        // remove subsequent arith.constant whose argument value non-zero
        // add c4_i32 as new arith.constant equal to 4
        // replace scf.index_switch and all cases with arith.cmpi eq 
        // remove shli
        // insert arith.select that outputs either 0 or 4 based on output of arith.cmpi

        func::FuncOp func= getOperation();
        auto arg0= func.getArguments()[0]; //stores arg0

        //here we walk over every operation defined inside our func and check if its a switchop
        func.walk([&](scf::IndexSwitchOp op){
            
            //first we need to store c0_i32
            auto selector= op.getArg(); //get value of arg0 (here %0)
            auto loc=op.getLoc();
            //auto caseVal= op.getCaseValues()[0]; //int value of ith case i.e. 0 here
     
            auto& case0Reg= op.getCaseRegions()[0].front();//return body block of case0
            auto& defaultReg= op.getDefaultRegion().front();//return body block of default
            auto outTrue=(dyn_cast<scf::YieldOp>(case0Reg.getTerminator())).getOperand(0); //access the scf.yield in this block and we save c0_i32
            auto outFalse=(dyn_cast<scf::YieldOp>(defaultReg.getTerminator())).getOperand(0); //save arg0

            //now at this location we want to insert cmpi and select with all info we gathered
            OpBuilder builder(op);
            auto zeroConst=builder.create<arith::ConstantIntOp>(loc, 0,32); //compare arg0 to 0
            auto cmpi= builder.create<arith::CmpIOp>(loc,arith::CmpIPredicate::eq, arg0, zeroConst);
            auto select= builder.create<arith::SelectOp>(loc,cmpi.getResult(),outTrue,outFalse);

            op.getResult(0).replaceAllUsesWith(select.getResult());
            op.erase();

        });
        //so we replace shli with new arith.constant(4) instruction
        func.walk([&](arith::ShLIOp shliOp){
            Value LHS= shliOp.getLhs();
            Value RHS= shliOp.getRhs();
            //auto firstArg= LHS.getDefiningOp<scf::IndexSwitchOp>(); //check is first arg is from output of indexswitch op
            //if !firstArg{
            //    return;
            //}
            auto locShli= shliOp.getLoc();
            OpBuilder builder(shliOp);


            auto shiftAmt= dyn_cast_or_null<arith::ConstantIntOp>(RHS.getDefiningOp());
            if (!shiftAmt){
                return;
            }
            auto shiftAmtVal= shiftAmt.value();

            if (shiftAmtVal==2){
                auto select=dyn_cast_or_null<arith::SelectOp>(LHS.getDefiningOp());
                if (!select){
                    return;
                }
                auto fourConst=builder.create<arith::ConstantIntOp>(locShli, 4,32); //compare arg0 to 0
                auto selL=select.getTrueValue().getDefiningOp<arith::ConstantIntOp>();
                auto selR=select.getFalseValue().getDefiningOp<arith::ConstantIntOp>();
                Value selTrue;
                Value selFalse;
            
                if (selL && selL.value()==0){
                    selTrue= select.getTrueValue();
                } else{
                    selTrue= fourConst;
                }

                if (selR && selR.value()==0){
                    selFalse= select.getFalseValue();
                } else{
                    selFalse= fourConst;
                }
                //auto selTrue= builder.create<arith::ShLIOp>(locShli,select.getTrueValue(),RHS);
                //auto selFalse= builder.create<arith::ShLIOp>(locShli,select.getFalseValue(),RHS);
                //auto fourConst=builder.create<arith::ConstantIntOp>(locShli,4,32);
                auto newselect= builder.create<arith::SelectOp>(locShli,select.getCondition(),selTrue,selFalse);

                shliOp.replaceAllUsesWith(newselect.getResult());
                shliOp.erase();
                return;
            }
        
            auto select=dyn_cast_or_null<arith::SelectOp>(LHS.getDefiningOp());
            if (!select){
                return;
            }

            auto selL=select.getTrueValue().getDefiningOp<arith::ConstantIntOp>();
            auto selR=select.getFalseValue().getDefiningOp<arith::ConstantIntOp>();
            Value selTrue;
            Value selFalse;
            
            if (selL && selL.value()==0){
                selTrue= select.getTrueValue();
            } else{
                selTrue= builder.create<arith::ShLIOp>(locShli,select.getTrueValue(),RHS);
            }

            if (selR && selR.value()==0){
                selFalse= select.getFalseValue();
            } else{
                selFalse= builder.create<arith::ShLIOp>(locShli,select.getFalseValue(),RHS);
            }
            //auto selTrue= builder.create<arith::ShLIOp>(locShli,select.getTrueValue(),RHS);
            //auto selFalse= builder.create<arith::ShLIOp>(locShli,select.getFalseValue(),RHS);
            //auto fourConst=builder.create<arith::ConstantIntOp>(locShli,4,32);
            auto newselect= builder.create<arith::SelectOp>(locShli,select.getCondition(),selTrue,selFalse);

            shliOp.replaceAllUsesWith(newselect.getResult());
            shliOp.erase();
            

        });
        //now we have to remove all arith.indexcast not used
        func.walk([&](arith::IndexCastOp castOp){
            if (castOp.use_empty()){
                castOp.erase();
            }
        });
        //now we have to remove all arith.constant not used
        func.walk([&](arith::ConstantOp constOp){
            if (constOp.use_empty()){
                constOp.erase();
            }
        });
        //adding greedy folding at end
        RewritePatternSet patterns(func.getContext());
        GreedyRewriteConfig config;
        (void)applyPatternsAndFoldGreedily(func, std::move(patterns), config);
};

};
}

int main(int argc, char **argv) {
  mlir::registerAllPasses();
  mlir::PassRegistration<mlir::InstCombinePass>();

  mlir::DialectRegistry registry;
  registerAllDialects(registry);

  return mlir::asMainReturnCode(
      mlir::MlirOptMain(argc, argv, "Custom optimizer driver\n", registry));
}
